# Add Payment to Silver

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DecimalType
)

In [0]:
import dlt as dp

## Variables

## Schema Definition

In [0]:
schema = StructType(
    [
        StructField(
            name="payment_type",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment":"Shows the RateCodeID"}
        ),
        StructField(
            name="payment_description",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Payment Code in Plain English"}
        ),
    ]
)

## ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.silver.payment_stm_payment",
    # Beschreibung der Tabelle
    comment="This table shows the available payment options",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    schema=schema,
)
def payment_stm_payment():
    df = spark.read.table("analytics.bronze.dwh_nyc_taxi_payment")
    df = df.withColumnRenamed("payment_desc", "payment_description")

    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
        
    return df